# More EDA on what should be cleaned (silver layer)

In [0]:
df = spark.sql("SELECT * FROM marathos.bronze.raw_marathos")



In [0]:
df.printSchema()

### Specific null counts

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

### Rename Columns 

In [0]:
import re

def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_colummns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

df_cleaned = rename_colummns_to_snake_case(df)

df_cleaned.display()


## Change "Event dates" to timestamps

In [0]:
df_cleaned.select("event_dates").display()

In [0]:
from pyspark.sql.functions import expr

# Byt ut kommatecken mot punkter och kör TRY_CAST för att säkert göra om det till Double
df_cleaned = df_cleaned.withColumn(
    "athlete_average_speed",
    expr("try_cast(regexp_replace(athlete_average_speed, ',', '.') AS DOUBLE)")
)
df_cleaned = df_cleaned.filter(
    (col("athlete_average_speed") > 0) & 
    (col("athlete_average_speed") <= 25)
)

df_cleaned.select("athlete_average_speed").display()


In [0]:
# Fick ta lite hjälp av llm med denna.
from pyspark.sql.functions import col, regexp_replace, expr

df_cleaned = df_cleaned.withColumn(
    "event_dates_clean", 
    regexp_replace(col("event_dates"), r"\.?-[0-9]+", "")
)

df_date = df_cleaned.withColumn(
    "event_date", 
    expr("try_to_date(event_dates_clean, 'dd.MM.yyyy')")
)

df_date.select("event_dates", "event_dates_clean", "event_date").display()

In [0]:
df_date.filter(col("event_date").isNull()).count()

In [0]:
df_cleaned = df_date.dropna(subset=["event_date"])


In [0]:
df_cleaned.filter(col("event_date").isNull()).select("event_dates", "event_dates_clean", "event_date").display()

## Check validity of athlete_performance

In [0]:
# Validate that if "event_distance/length" contains ":" then "athlete_performance" should end in "h"
# If not then it should end in "km" or "mi"
from pyspark.sql.functions import col, when

df_validated = df_cleaned.withColumn(
    "is_valid_performance",
    when(
        col("event_distance/length").contains(":"), 
        col("athlete_performance").endswith("h")
    ).otherwise(
        col("athlete_performance").endswith("km") | col("athlete_performance").endswith("mi")
    )
)

df_filtered = df_validated.filter(col("is_valid_performance") == True)

df_clean_performance = df_filtered.drop("is_valid_performance")

## Cleaning "event_distance/length"

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, lower, when, expr

df_cleaned = df_cleaned.filter(~col("event_distance/length").contains("/"))
df_cleaned = df_cleaned.withColumn(
    "event_distance/length", 
    expr("regexp_replace(`event_distance/length`, ',', '.')")
)

df_cleaned = df_cleaned.withColumn(
    "raw_unit", 
    lower(trim(regexp_replace(col("event_distance/length"), r"[0-9\.]", "")))
)

df_cleaned = df_cleaned.withColumn(
    "event_distance_unit",
    when(col("raw_unit").isin(["miles", "mile", "mi+", "mi"]), "mi")
    .when(col("raw_unit").isin(["k", "km"]), "km")
    .when(col("raw_unit").isin([":h", "h"]), "h")
    .otherwise("invalid") 
)

# Kasta bort allt invalid
df_cleaned = df_cleaned.filter(col("event_distance_unit") != "invalid")

df_cleaned.select("event_distance_unit").distinct().display()

In [0]:
df_cleaned.count()

In [0]:
df_cleaned.display()

### Convert time of athlete performance to correct datatype

In [0]:
from pyspark.sql.functions import col, when, regexp_replace, trim, split, size, lit, round
# Behövde ta hjälp av llm med denna 

df_cleaned = df_cleaned.withColumn(
    "athlete_performance_clean", 
    trim(regexp_replace(regexp_replace(col("athlete_performance"), ",", "."), r"[a-zA-Z\s]", ""))
)

df_cleaned = df_cleaned.withColumn(
    "time_array", 
    split(col("athlete_performance_clean"), ":")
)

# Plockar ut timmarna, minuter å sekunder
hours = col("time_array").getItem(0).cast("double")
minutes = when(size(col("time_array")) > 1, col("time_array").getItem(1).cast("double")).otherwise(lit(0.0))
seconds = when(size(col("time_array")) > 2, col("time_array").getItem(2).cast("double")).otherwise(lit(0.0))

df_cleaned = df_cleaned.withColumn(
    "athlete_performance_value",
    when(
        col("athlete_performance_clean").contains(":"),
        round(hours + (minutes / 60) + (seconds / 3600), 2)
    ).otherwise(
        round(col("athlete_performance_clean").cast("double"), 2)
    )
)

df_cleaned = df_cleaned.drop("athlete_performance_clean", "time_array")


df_cleaned = df_cleaned.withColumn(
    "performance_unit",
    when(col("event_distance_unit").isin(["km", "mi"]), "h")
    .when(col("athlete_performance").contains("mi"), "mi")
    .otherwise("km") 
)

df_cleaned.select(
    "event_distance/length",
    "athlete_performance", 
    "athlete_performance_value", 
    "performance_unit", 
    "event_distance_unit"
).display()

## Check Age category group

In [0]:
df_cleaned.select("athlete_age_category").distinct().display()

### Delete all nulls except "athlete_club" 

In [0]:
all_columns = df_cleaned.columns

# Cleara alla nulls förutom "athlete club" eftersom det är ok att springa utan en klubb
columns_to_clear = [col_name for col_name in all_columns if col_name != "athlete_club"]

df_cleaned = df_cleaned.dropna(subset=columns_to_clear)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_cleaned.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df_cleaned.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

### Create new column "event_country" from event_name

In [0]:
# Take last values of "event_name" and save to new column "event_country"

from pyspark.sql.functions import col, regexp_extract, trim, regexp_replace

df_cleaned = df_cleaned.withColumn(
    "event_country",
    trim(regexp_extract(col("event_name"), r"\(([^)]+)\)\s*$", 1))
)

df_cleaned.select("event_name", "event_country").display()

In [0]:
df_cleaned.select("year_of_event").distinct().orderBy("year_of_event").display()

### DataType conversions

In [0]:
# Athlete year of birth should be an int and not double
df_cleaned = df_cleaned.withColumn(
    "athlete_year_of_birth",
    col("athlete_year_of_birth").cast("int")
)
df_cleaned.select("athlete_year_of_birth").display()

In [0]:
df_cleaned.columns.display()